<a href="https://colab.research.google.com/github/spirosChv/neuro208-tutorials/blob/main/7_NeuroAI/Notebook5_image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spiking Neural Network with Surrogate Gradients

This notebook implements a **Current-Based Leaky Integrate-and-Fire (CUBA-LIF)** spiking neural network trained on MNIST using **surrogate gradients**.

In [ ]:
!pip install mplcyberpunk -q

In [ ]:
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import mplcyberpunk

## Surrogate Gradient Function

Spiking neurons use a Heaviside step function to generate spikes, which has a zero gradient almost everywhere. To enable backpropagation, we replace the gradient in the backward pass with a smooth **surrogate** — here, the derivative of the arctangent.

In [ ]:
class ATanSurrogate(torch.autograd.Function):
    """
    A surrogate gradient function using the derivative of the arctangent. This
    custom autograd function implements the Heaviside step function in the
    forward pass and its surrogate gradient (based on the arctangent function's
    derivative) in the backward pass. The steepness of the gradient is
    controlled by the `alpha` parameter.
    """
    @staticmethod
    def forward(ctx, x, alpha=2.0):
        """
        Applies the Heaviside step function in the forward pass.

        Args:
            ctx: A context object to save information for the backward pass.
            x (torch.Tensor): The input tensor (e.g., membrane potential minus threshold).
            alpha (float, optional): The steepness parameter for the gradient. Defaults to 2.0.

        Returns:
            torch.Tensor: A binary tensor, where elements are 1.0 if the corresponding
                          input element is greater than 0, and 0.0 otherwise.
        """
        ctx.save_for_backward(x)
        ctx.alpha = alpha
        return (x > 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        """
        Computes the gradient using the arctangent derivative surrogate.

        Args:
            ctx: The context object with saved tensors from the forward pass.
            grad_output (torch.Tensor): The gradient of the loss with respect to the
                                        output of the forward pass.

        Returns:
            Tuple[torch.Tensor, None]: A tuple containing the gradient with respect to the
                                       input `x`, and None for the `alpha` parameter as it
                                       does not require a gradient.
        """
        x, = ctx.saved_tensors
        alpha = ctx.alpha
        grad_x = (alpha / 2) / (1.0 + (torch.pi / 2 * alpha * x).pow(2))
        return grad_output * grad_x, None

## CUBA-LIF Neuron Model

The **Current-Based Leaky Integrate-and-Fire** neuron has two state variables:
- **Synaptic current** ($I_{syn}$): decays exponentially, driven by incoming spikes
- **Membrane potential** ($V_{mem}$): decays exponentially, driven by the synaptic current

When the membrane potential crosses a threshold, the neuron emits a spike and resets.

In [ ]:
class CUBAPointLeaky(nn.Module):
    """
    A Current-Based Leaky Integrate-and-Fire (CUBA-LIF) neuron.
    Spikes induce a decaying synaptic current, which then drives the membrane potential.
    """
    def __init__(
        self,
        threshold: float,
        reset: float,
        tau_mem: float,
        tau_syn: float,
        R: float,
        dt: float,
    ):
        super().__init__()
        self.threshold = threshold
        self.reset = reset

        # Precompute decay factors (alphas) using the exponential form
        self.alpha_mem = torch.exp(torch.tensor(-dt / tau_mem)) # membrane voltage
        self.alpha_syn = torch.exp(torch.tensor(-dt / tau_syn)) # synaptic current

        # Pre-calculate the voltage scaling term: R * (1 - alpha_mem)
        # This converts the total Current (Amperes) into a Voltage step.
        self.voltage_scale = R * (1.0 - self.alpha_mem)

    def initialize_state(self, batch_size: int, num_neurons: int, device: torch.device):
        """Initialize the state tuple (Membrane, Synaptic Current)"""
        shape = (batch_size, num_neurons)
        mem = torch.zeros(shape, device=device)
        isyn = torch.zeros(shape, device=device)
        return mem, isyn

    def forward(
        self,
        x,
        state: tuple[torch.Tensor, torch.Tensor],
        use_surrogate: bool = False,
        surrogate_alpha: float = 2.0
    ):
        """
        x: Incoming spikes (delta inputs)
        state: Tuple (mem, isyn)
        """
        mem, isyn = state

        # Update synaptic current
        new_isyn = self.alpha_syn * isyn + x

        # Update membrane potential
        new_mem = self.alpha_mem * mem + self.voltage_scale * new_isyn

        # Generate spikes using either surrogate gradient or hard threshold
        if use_surrogate:
            spike = ATanSurrogate.apply(new_mem - self.threshold, surrogate_alpha)
        else:
            spike = (new_mem > self.threshold).float()

        # Reset membrane potential where spikes occurred
        mem = new_mem * (1 - spike.detach()) + self.reset * spike.detach()

        return spike, (mem, new_isyn)

## Spiking Neural Network

A two-layer SNN: input spikes are generated via Bernoulli sampling from pixel intensities, passed through a hidden layer of LIF neurons, then to an output layer. The network counts output spikes over all timesteps as the classification signal.

In [ ]:
class CUBAPointSurrogateNet(nn.Module):
    def __init__(
            self,
            num_inputs: int,
            num_hidden: int,
            num_outputs: int,
            num_steps: int,
            threshold: float,
            reset: float,
            tau_mem: float,
            tau_syn: float,
            R: float,
            dt: float,
    ):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_hidden = num_hidden
        self.num_outputs = num_outputs
        self.num_steps = num_steps

        # ---- Initialize layers ----------------------------------------------#
        self.weights_hidden = nn.Linear(num_inputs, num_hidden)
        self.lif_hidden = CUBAPointLeaky(
            threshold=threshold,
            reset=reset,
            R=R,
            tau_mem=tau_mem,
            tau_syn=tau_syn,
            dt=dt)

        self.weights_out = nn.Linear(num_hidden, num_outputs)
        self.lif_out = CUBAPointLeaky(
            threshold=threshold,
            reset=reset,
            R=R,
            tau_mem=tau_mem,
            tau_syn=tau_syn,
            dt=dt)

    def forward(self, x):
        # Flatten input
        x = x.view(x.size(0), -1)
        B = x.size(0) # Batch size

        # Initialize membrane values at t=0
        mem_hidden, isyn_hidden = self.lif_hidden.initialize_state(
            batch_size=B,
            num_neurons=self.num_hidden,
            device=x.device)
        mem_out, isyn_out = self.lif_out.initialize_state(
            batch_size=B,
            num_neurons=self.num_outputs,
            device=x.device)

        # Create empty tensor to count output spikes
        spk_count_output = torch.zeros(B, self.num_outputs, device=x.device)

        # Iterate through each timestep
        for step in range(self.num_steps):
            # Generate input spikes for this timestep
            spk_input = torch.bernoulli(x)

            # Forward pass through the SNN
            cur_hidden = self.weights_hidden(spk_input)
            spk_hidden, (mem_hidden, isyn_hidden) = self.lif_hidden(
                cur_hidden, (mem_hidden, isyn_hidden), use_surrogate=True)
            cur_out = self.weights_out(spk_hidden)
            spk_out, (mem_out, isyn_out) = self.lif_out(
                cur_out, (mem_out, isyn_out), use_surrogate=True)

            # Accumulate output spikes
            spk_count_output += spk_out

        return spk_count_output

## Utility Functions

In [ ]:
def clean_dataset(dataset, max_devide=True, scale=1.0):
    """
    Extracts data and targets from the dataset and normalizes the data.
    Pre-loading into tensors significantly improves training performance.
    """
    X, y = dataset.data.unsqueeze(1).float(), dataset.targets.long()
    if max_devide:
        X /= X.max()
    X *= scale
    return torch.utils.data.TensorDataset(X, y)


def train_network(
        model,
        train_data,
        test_data,
        batch_size,
        epochs,
        optimizer_name,
        lr,
        device,
        print_progress=True,
        pin_memory=False,
        num_workers=0):

    # Create DataLoaders
    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_memory,
        num_workers=num_workers,
        multiprocessing_context='fork' if num_workers > 0 else None)

    test_loader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=num_workers,
        multiprocessing_context='fork' if num_workers > 0 else None)

    # Setup loss function and optimizer
    loss_fn = nn.CrossEntropyLoss()
    optimizer_params = {'params': model.parameters(), 'lr': lr}
    if optimizer_name == 'SGD':
        optimizer_params.update({'momentum': 0.9, 'weight_decay': 0.0})
    optimizer = getattr(torch.optim, optimizer_name)(**optimizer_params)

    # Main training and evaluation loop
    model.to(device)
    results = {'train_loss': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(epochs):
        epoch_start_time = time.time()

        # ---- Training Phase -------------------------------------------------#
        model.train()
        train_loss = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)
        results['train_loss'].append(train_loss)

        # ---- Testing Phase --------------------------------------------------#
        model.eval()
        test_loss, test_acc = 0, 0
        with torch.no_grad():
            for X_test, y_test in test_loader:
                X_test, y_test = X_test.to(device), y_test.to(device)
                test_pred = model(X_test)
                test_loss += loss_fn(test_pred, y_test).item()
                test_acc += (test_pred.argmax(dim=1) == y_test).sum().item()
        test_loss /= len(test_loader)
        test_acc = (test_acc / len(test_data)) * 100
        results['test_loss'].append(test_loss)
        results['test_acc'].append(test_acc)

        # ---- Progress Printing ----------------------------------------------#
        epoch_duration = time.time() - epoch_start_time
        if print_progress:
            print(
                f"Epoch: {epoch+1} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Test Loss: {test_loss:.4f} | "
                f"Test Acc: {test_acc:.2f}% | "
                f"Time: {epoch_duration:.2f}s")

    return results


def plot_results(train_losses, test_losses, test_accuracies, save=False):
    """Plot training and test losses along with test accuracies."""
    if torch.is_tensor(train_losses):
        train_losses = train_losses.cpu().numpy()
    if torch.is_tensor(test_losses):
        test_losses = test_losses.cpu().numpy()
    if torch.is_tensor(test_accuracies):
        test_accuracies = test_accuracies.cpu().numpy()

    with plt.style.context("cyberpunk"):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

        # Left panel: Train and test loss
        ax1.plot(train_losses, label='Train')
        ax1.plot(test_losses, label='Test')
        ax1.set_xlabel('Epoch', color='white')
        ax1.set_ylabel('Loss', color='white')
        ax1.set_title('Loss Curves', color='white')
        ax1.tick_params(colors='white')
        ax1.legend()
        mplcyberpunk.add_glow_effects(ax1)

        # Right panel: Test accuracy
        ax2.plot(test_accuracies, label='Test', color='#00ff41')
        ax2.set_xlabel('Epoch', color='white')
        ax2.set_ylabel('Accuracy (%)', color='white')
        ax2.set_title('Accuracy Curves', color='white')
        ax2.tick_params(colors='white')
        ax2.legend()
        mplcyberpunk.add_glow_effects(ax2)

        fig.tight_layout()
        plt.show()
        if save:
            fig.savefig("training_results.png", dpi=300)

## Configuration & Training

Set the hyperparameters below and run the training loop.

In [ ]:
# Simulation
DT = 1e-3         # time step in seconds
TIME = 30e-3       # image presentation time in seconds
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Neuron properties
THRESHOLD = 1.0
RESET = 0.0
R = 5
TAU_MEM = 10e-3    # membrane time constant in seconds
TAU_SYN = 2e-3     # synaptic time constant in seconds

# Network
NUM_INPUTS = 784
NUM_HIDDEN = 128
NUM_OUTPUTS = 10
INPUT_SCALE = 1.0  # max probability of input spikes

# Training
DATASET = 'MNIST'
BATCH_SIZE = 256
EPOCHS = 10
OPTIMIZER = 'Adam'
LR = 1e-3
WORKERS = 2

In [ ]:
device = torch.device(DEVICE)
torch.manual_seed(SEED)

# Create the network
num_steps = int(TIME / DT)
network = CUBAPointSurrogateNet(
    num_inputs=NUM_INPUTS,
    num_hidden=NUM_HIDDEN,
    num_outputs=NUM_OUTPUTS,
    num_steps=num_steps,
    threshold=THRESHOLD,
    reset=RESET,
    R=R,
    tau_mem=TAU_MEM,
    tau_syn=TAU_SYN,
    dt=DT)

# Load and prepare the dataset
train_data = getattr(datasets, DATASET)(root='./data', train=True, download=True, transform=ToTensor())
test_data = getattr(datasets, DATASET)(root='./data', train=False, download=True, transform=ToTensor())

dt_input_scale = INPUT_SCALE * (DT / 1e-3)  # scale down bernoulli prob when using smaller time steps
train_data = clean_dataset(train_data, scale=dt_input_scale)
test_data = clean_dataset(test_data, scale=dt_input_scale)

print(f"Device: {device}")
print(f"Network timesteps: {num_steps}")
print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")

In [ ]:
# Save initial weights before training
weights_before = {
    name: param.data.clone().flatten().cpu().numpy()
    for name, param in network.named_parameters()
    if 'weight' in name
}

In [ ]:
# Train the network
t_start = time.time()
results = train_network(
    network,
    train_data,
    test_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    optimizer_name=OPTIMIZER,
    lr=LR,
    device=device,
    num_workers=WORKERS,
    pin_memory=DEVICE == "cuda")
t_end = time.time()

# Report results
best_test_acc = max(results['test_acc'])
best_epoch = results['test_acc'].index(best_test_acc)
print(f"\nBest test acc: {best_test_acc:.2f}% at epoch {best_epoch + 1}")
print(f"Total training time: {t_end - t_start:.2f}s")

In [ ]:
plot_results(results['train_loss'], results['test_loss'], results['test_acc'])

In [ ]:
# Collect post-training weights
weights_after = {
    name: param.data.clone().flatten().cpu().numpy()
    for name, param in network.named_parameters()
    if 'weight' in name
}

w_hidden_before = weights_before['weights_hidden.weight']
w_hidden_after  = weights_after['weights_hidden.weight']
w_output_before = weights_before['weights_out.weight']
w_output_after  = weights_after['weights_out.weight']

with plt.style.context("cyberpunk"):
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    # Top row: Hidden layer
    axes[0, 0].hist(w_hidden_before, bins=50, color='#08F7FE', edgecolor='none', alpha=0.8)
    axes[0, 0].set_title('Hidden weights — Before training', color='white')
    axes[0, 0].set_ylabel('Count', color='white')
    axes[0, 0].tick_params(colors='white')
    mplcyberpunk.add_glow_effects(axes[0, 0])

    axes[0, 1].hist(w_hidden_after, bins=50, color='#FE53BB', edgecolor='none', alpha=0.8)
    axes[0, 1].set_title('Hidden weights — After training', color='white')
    axes[0, 1].tick_params(colors='white')
    mplcyberpunk.add_glow_effects(axes[0, 1])

    # Bottom row: Output layer
    axes[1, 0].hist(w_output_before, bins=30, color='#08F7FE', edgecolor='none', alpha=0.8)
    axes[1, 0].set_title('Output weights — Before training', color='white')
    axes[1, 0].set_xlabel('Weight value', color='white')
    axes[1, 0].set_ylabel('Count', color='white')
    axes[1, 0].tick_params(colors='white')
    mplcyberpunk.add_glow_effects(axes[1, 0])

    axes[1, 1].hist(w_output_after, bins=30, color='#FE53BB', edgecolor='none', alpha=0.8)
    axes[1, 1].set_title('Output weights — After training', color='white')
    axes[1, 1].set_xlabel('Weight value', color='white')
    axes[1, 1].tick_params(colors='white')
    mplcyberpunk.add_glow_effects(axes[1, 1])

    fig.suptitle('Weight distributions', color='white', fontsize=14)
    fig.tight_layout()
    plt.show()

In [ ]:
# Pick 9 random test samples
indices = torch.randperm(len(test_data))[:9]

images = test_data.tensors[0][indices]   # (9, 1, 28, 28)
labels = test_data.tensors[1][indices]   # (9,)

# Get predictions
network.eval()
with torch.no_grad():
    preds = network(images.to(device)).argmax(dim=1).cpu()

with plt.style.context("cyberpunk"):
    fig, axes = plt.subplots(3, 3, figsize=(7, 7))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i].squeeze(), cmap='gray')
        correct = preds[i].item() == labels[i].item()
        color = '#00ff41' if correct else 'red'
        ax.set_title(f'Pred: {preds[i].item()} | True: {labels[i].item()}',
                     fontsize=11, color=color)
        ax.axis('off')
    fig.suptitle('Model Predictions on Random Test Samples', fontsize=14, color='white', y=1.02)
    fig.tight_layout()
    plt.show()